# Week 7: Pair Programming - Regularization Detective

**Exercise:** Diagnose and fix overfitting model  
**Time:** 20 minutes  
**Scaffolding:** 60% provided (diagnostic focus)  
**Partners:** Work in pairs - switch roles at 10 minutes!

---

## Your Mission

You're receiving an **intentionally overfitting** Fashion-MNIST model. Your job:

1. Run the bad model → Plot curves → Diagnose the problem
2. Add Dropout layers to fix it
3. Add EarlyStopping callback
4. Re-train → Compare before/after
5. Save your improved model

**Partner Roles:**
- **Driver:** Types code
- **Navigator:** Guides decisions and checks understanding
- **Switch at 10 minutes!**

---

## Setup (Provided)

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
from keras import layers
from keras.callbacks import EarlyStopping
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

print("Setup complete!")

## Load Data (Provided)

In [ ]:
# Load Fashion-MNIST
from keras.datasets import fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

# Normalize
X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

# Three-way split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

# Flatten
X_train_flat = X_train.reshape(-1, 784)
X_val_flat = X_val.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)

print(f"Training: {X_train_flat.shape}")
print(f"Validation: {X_val_flat.shape}")
print(f"Test: {X_test_flat.shape}")
print("Data ready!")

---

## Part 1: Run Overfitting Model (Provided)

This model is **intentionally bad** - too large, no regularization!

In [ ]:
# BAD MODEL - No regularization!
model_bad = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_bad.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Overfitting model architecture:")
model_bad.summary()

In [ ]:
# Train the bad model (this will overfit!)
print("Training overfitting model...")
history_bad = model_bad.fit(
    X_train_flat, y_train,
    validation_data=(X_val_flat, y_val),
    epochs=20,
    batch_size=128,
    verbose=0  # Quiet training
)

print("Training complete!")

### Task 1: Plot Training Curves and Diagnose

**TODO:** Plot the training curves and identify the overfitting pattern

In [ ]:
# TODO: Plot training and validation curves
# HINT: Use history_bad.history['loss'] and history_bad.history['val_loss']

plt.figure(figsize=(12, 4))

# Loss plot
plt.subplot(1, 2, 1)
# TODO: Plot training loss
plt.plot(history_bad.history['____'], label='Training Loss', linewidth=2)
# TODO: Plot validation loss
plt.plot(history_bad.history['____'], label='Validation Loss', linewidth=2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss - Bad Model')
plt.grid(True, alpha=0.3)

# Accuracy plot
plt.subplot(1, 2, 2)
# TODO: Plot training accuracy
plt.plot(history_bad.history['____'], label='Training Accuracy', linewidth=2)
# TODO: Plot validation accuracy
plt.plot(history_bad.history['____'], label='Validation Accuracy', linewidth=2)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy - Notice the Gap!')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final performance
train_acc_bad = history_bad.history['accuracy'][-1]
val_acc_bad = history_bad.history['val_accuracy'][-1]
print(f"\nBad Model Performance:")
print(f"Training Accuracy: {train_acc_bad:.4f}")
print(f"Validation Accuracy: {val_acc_bad:.4f}")
print(f"Gap (overfitting): {(train_acc_bad - val_acc_bad):.4f}")

**Discuss with partner:**
- What pattern do you see in the curves?
- Which Training Curves Pattern does this match?
- Is training accuracy much higher than validation accuracy?
- **Diagnosis: ________________________________**

---

## Part 2: Fix with Dropout

**TODO:** Add Dropout layers to prevent overfitting

In [ ]:
# TODO: Build improved model with Dropout
# HINTS:
# - Add Dropout after each Dense layer (except output)
# - Try rate=0.3 (drops 30% of neurons)
# - NO Dropout after the output layer!

model_good = keras.Sequential([
    layers.Input(shape=(784,)),
    
    layers.Dense(512, activation='relu'),
    # TODO: Add Dropout layer here
    layers.Dropout(____),
    
    layers.Dense(256, activation='relu'),
    # TODO: Add Dropout layer here
    layers.Dropout(____),
    
    layers.Dense(128, activation='relu'),
    # TODO: Add Dropout layer here
    layers.Dropout(____),
    
    layers.Dense(10, activation='softmax')  # NO Dropout here!
])

print("Improved model with Dropout:")
model_good.summary()

**Discuss with partner:**
- Why add Dropout after Dense layers?
- Why NOT add Dropout after output layer?
- What does rate=0.3 mean?

---

## Part 3: Add EarlyStopping Callback

**TODO:** Configure EarlyStopping to stop training automatically

In [ ]:
# Compile model
model_good.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# TODO: Create EarlyStopping callback
# HINTS:
# - monitor='val_loss' (watch validation loss)
# - patience=3 or 5 (how many epochs to wait)
# - restore_best_weights=True (go back to best epoch)

early_stop = EarlyStopping(
    monitor='____',           # What to watch?
    patience=____,            # How long to wait?
    restore_best_weights=____, # True or False?
    verbose=1
)

print("EarlyStopping configured!")

**Discuss with partner:**
- What does patience=3 mean?
- Why monitor 'val_loss' instead of 'loss'?
- Why restore_best_weights?

---

## SWITCH ROLES NOW!
**Driver becomes Navigator, Navigator becomes Driver**

---

## Part 4: Train Improved Model

Train with Dropout + EarlyStopping

In [ ]:
# Train improved model
print("Training improved model with regularization...")
history_good = model_good.fit(
    X_train_flat, y_train,
    validation_data=(X_val_flat, y_val),  # Required for EarlyStopping!
    epochs=50,  # Set high - EarlyStopping will stop us
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

print("\nTraining complete (stopped early!)")

### Plot Improved Curves

In [ ]:
# Plot improved training curves
plt.figure(figsize=(12, 4))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(history_good.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history_good.history['val_loss'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss - Improved Model')
plt.grid(True, alpha=0.3)

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(history_good.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history_good.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy - Much Better!')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print improved performance
train_acc_good = history_good.history['accuracy'][-1]
val_acc_good = history_good.history['val_accuracy'][-1]
print(f"\nImproved Model Performance:")
print(f"Training Accuracy: {train_acc_good:.4f}")
print(f"Validation Accuracy: {val_acc_good:.4f}")
print(f"Gap (much smaller!): {(train_acc_good - val_acc_good):.4f}")

**Discuss with partner:**
- Are the curves closer together now?
- What Training Curves Pattern do you see?
- Did EarlyStopping trigger? At what epoch?

---

## Part 5: Compare Before and After

In [ ]:
# TODO: Compare validation accuracy improvement

print("\n" + "="*50)
print("BEFORE vs AFTER Comparison")
print("="*50)

print(f"\nBAD MODEL (no regularization):")
print(f"  Training Accuracy: {train_acc_bad:.4f}")
print(f"  Validation Accuracy: {val_acc_bad:.4f}")
print(f"  Gap: {(train_acc_bad - val_acc_bad):.4f}")

print(f"\nGOOD MODEL (Dropout + EarlyStopping):")
print(f"  Training Accuracy: {train_acc_good:.4f}")
print(f"  Validation Accuracy: {val_acc_good:.4f}")
print(f"  Gap: {(train_acc_good - val_acc_good):.4f}")

# TODO: Calculate improvement
improvement = val_acc_good - val_acc_bad
print(f"\nValidation Accuracy Improvement: {improvement:+.4f}")

if improvement > 0:
    print("SUCCESS! Regularization improved validation performance!")
else:
    print("Hmm, try adjusting dropout rate or patience?")

---

## Part 6: Save Your Improved Model

In [ ]:
# TODO: Save the improved model
# HINT: Use model_good.save() with .keras extension

model_good.save('____')
print("Improved model saved!")

# Verify by loading
from keras.models import load_model
loaded = load_model('fashion_mnist_improved.keras')
print("Model loaded successfully - ready for deployment!")

---

## Reflection Questions

**Discuss with your partner:**

1. **Diagnosis:** What overfitting pattern did you see in the bad model?
   - _______________________________________________________________________

2. **Dropout:** How did adding Dropout help?
   - _______________________________________________________________________

3. **EarlyStopping:** At what epoch did training stop? Was it before epoch 50?
   - _______________________________________________________________________

4. **Improvement:** Did validation accuracy improve? By how much?
   - _______________________________________________________________________

5. **Real World:** When would you use these techniques in production?
   - _______________________________________________________________________

---

## Bonus Challenges (If Time)

1. **Try different dropout rates:** 0.2, 0.4, 0.5 - which works best?
2. **Experiment with patience:** Try patience=3 vs patience=7 - what changes?
3. **Test set evaluation:** Evaluate both models on test set - which generalizes better?
4. **Smaller model:** Try 256→128→64 architecture - does it still overfit?

---

## Key Takeaways

**You just completed the Week 7 workflow:**

1. Diagnose problem (plot curves, identify overfitting)
2. Apply fixes (Dropout + EarlyStopping)
3. Verify improvement (compare before/after)
4. Save for production (model.save())

**This is what data scientists do every day!**

Great work!

---

*Week 7 Pair Programming | Version 1.0 | February 2026*